# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata (name and description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each record set and field is uniquely identified by its `@id`. Below, we inspect the main dataset structure.

In [ ]:
# List all record sets and their @id in the dataset
record_sets = list(dataset.metadata.record_sets)
if not record_sets:
    print("No record sets found in this Croissant schema.")
else:
    print("Available record sets and their IDs:\n")
    for rs in record_sets:
        print(f"- Name: {rs.name}, @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            col_str = f" (@id: {field.id})"
            print(f"    - {field.name}{col_str}")
        print()
    # Store the first record set for further extraction
    first_record_set_id = record_sets[0].id

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set (by @id)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for {record_set_id}")
        print(f"Columns: {dataframes[record_set_id].columns.tolist()}")
    else:
        print(f"No records found for {record_set_id}.")

# Preview the first few rows of each DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nPreview for record set: {record_set_id}")
    display(df.head())


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** All fields are referenced by their `@id`. You may need to examine the Print outputs above for the exact `@id` of each field.

Below, we demonstrate EDA on the first available record set.

In [ ]:
import numpy as np

# Select one record set to analyze (first available)
if not dataframes:
    print("No dataframes available for analysis.")
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Analyzing record set: {record_set_id}")

    # Try to identify numeric fields (by dtype or previewing names)
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_fields:
        print("No numeric fields detected. Displaying DataFrame for manual inspection.")
        display(df.head())
    else:
        numeric_field_id = numeric_fields[0]  # Choose the first numeric field by column name (@id)
        print(f"Using numeric field: {numeric_field_id}")

        threshold = df[numeric_field_id].mean()  # For demonstration, use mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} ({len(filtered_df)} records):")
        display(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try finding a non-numeric/grouping field (by exclusion)
        group_fields = [col for col in df.columns if col not in numeric_fields]
        if group_fields:
            group_field_id = group_fields[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field was found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f'Distribution of {numeric_field_id}')
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.show()

        # If another numeric column is available, show a scatter plot
        if len(numeric_fields) > 1:
            numeric_field_id2 = numeric_fields[1]
            plt.figure(figsize=(7, 5))
            sns.scatterplot(data=df, x=numeric_field_id, y=numeric_field_id2)
            plt.title(f'{numeric_field_id} vs {numeric_field_id2}')
            plt.xlabel(numeric_field_id)
            plt.ylabel(numeric_field_id2)
            plt.show()
    else:
        print("No numeric fields for plotting.")
else:
    print("No available data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Inspected dataset structure via Croissant schema.
- Loaded metadata and records using `mlcroissant`, referencing all entities by their `@id`.
- Previewed and analyzed fields, performed basic data normalization, and visualized data distributions.
- The approach in this notebook can be extended for more advanced analysis, such as hierarchical subsetting, advanced statistics, or machine learning workflows.

----

_Note: To adapt this notebook for other datasets, ensure you reference entities using their Croissant `@id` fields!_